# Volatility Dynamics 10b — model-free physical benchmark for 0DTE wings

**VD 021.** Module 10 found the wings RICH against a constant-volatility + Poisson-jump model, but that model omits variance uncertainty and multi-bar moves. 10b removes the model.

**Standardized move to settlement.** For every asset and every 5-minute origin (60–1200 min to the 08:00 expiry):

$$u_t=\frac{\ln(S_T/I_t)}{\sqrt{D_t}}$$

Here $D_t$ is exactly Module 10's ex-ante variance. Quotes and outcomes share the same $D$, so any level bias in $D$ cancels.

**Empirical price.** Each quote is priced as

$$P^{emp}=\textstyle\sum_i w_i\,\text{payoff}\big(F e^{\sqrt D u_i}/\bar m,K\big)$$

using the leave-one-session-out pool of $u$ (raw outcomes in the outer 1% tails, bin means in the centre), with a per-quote martingale normalization $\bar m$.

**Pre-registered (VD 021):**
- **Rows:** the same as Module 10's primary sample ($z\in[2,5)$, non-floor), read from `10/10_priced_quotes.csv.gz`.
- **Statistic:** $M$ = the median over sessions of the session median $\ln(\text{mid}/P^{emp})$.
- **Confidence interval:** a **joint** session bootstrap (quotes and outcome pool resampled together, B = 1000).
- **Verdict:** RICH / CHEAP / CONSISTENT.
- **Zero-price rows:** rows with $P^{emp}=0$ (strike beyond every observed outcome) are dropped, which is conservative against RICH. INSUFFICIENT_TAIL applies if more than 25% are dropped.

**Reported only:** the TTE-conditioned pool, the variance mixture $P^{mix}$, per-asset results, buckets and tail frequencies. Outputs: `10b/`.

In [1]:
from pathlib import Path
import json, os, sys, time, warnings
import numpy as np
import pandas as pd
import duckdb
import matplotlib.pyplot as plt
from dotenv import load_dotenv

warnings.filterwarnings('ignore', category=RuntimeWarning)
warnings.filterwarnings('ignore', category=FutureWarning)
pd.set_option('display.max_columns', 200)
pd.set_option('display.width', 300)
pd.set_option('display.max_rows', 300)

load_dotenv()
DATA_ROOT = Path(os.environ['DERIBIT_DATA_ROOT']).expanduser().resolve()
DB_PATH = DATA_ROOT / 'database' / 'research.duckdb'
PROJECT_ROOT = Path.cwd().resolve()
while PROJECT_ROOT != PROJECT_ROOT.parent and not (PROJECT_ROOT / 'src' / 'data_engineering').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / 'src' / 'data_engineering').exists():
    raise RuntimeError('Repository root not found.')
SRC = PROJECT_ROOT / 'src'
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from data_engineering.modeling import volatility_vd_b_core as vb
from data_engineering.modeling import volatility_jump_compound as jc
from data_engineering.modeling import volatility_jump_option_wings as ow
from data_engineering.modeling import volatility_wings_empirical as we
START = os.getenv('VD10B_START', '').strip() or None
END = os.getenv('VD10B_END', '').strip() or None
ASSETS = [x.strip().upper() for x in os.getenv('VD10B_ASSETS', '').split(',') if x.strip()]
N_BOOT = int(os.getenv('VD10B_N_BOOT', str(we.N_BOOT)))
N_BOOT_MIX = int(os.getenv('VD10B_N_BOOT_MIX', '300'))
OUT = PROJECT_ROOT / 'notebooks' / 'Volatility_Dynamics' / '10b'
OUT.mkdir(parents=True, exist_ok=True)
IN10 = PROJECT_ROOT / 'notebooks' / 'Volatility_Dynamics' / '10' / '10_priced_quotes.csv.gz'
print('Module 10b', we.VERSION, we.selftest())


Module 10b 0.1.0 {'gaussian_matches_b76_z2': True, 'mixture_k1_is_b76': True, 'bin_means_preserve_mean': True, 'fat_world_correct_benchmark_consistent': True, 'fat_world_gaussian_benchmark_rich': True, 'loso_excludes_own_session': True, 'outcome_u_sd_near_1': True, 'outcome_k_median_near_1': True, 'n_outcomes': 10074}


In [2]:
def qident(name):
    return '"' + str(name).replace('"', '""') + '"'

def load_frame(columns, start=None, end=None, assets=None):
    cols = list(dict.fromkeys(['state_time', 'underlying', 'return_5m'] + list(columns)))
    where = []
    if start:
        ts = pd.Timestamp(start)
        ts = ts.tz_localize('UTC') if ts.tzinfo is None else ts.tz_convert('UTC')
        where.append(f"state_time >= TIMESTAMPTZ '{ts.isoformat()}'")
    if end:
        ts = pd.Timestamp(end)
        ts = ts.tz_localize('UTC') if ts.tzinfo is None else ts.tz_convert('UTC')
        where.append(f"state_time < TIMESTAMPTZ '{(ts + pd.Timedelta(days=1)).isoformat()}'")
    if assets:
        where.append('underlying IN (' + ','.join(repr(x) for x in assets) + ')')
    sql = ('SELECT ' + ', '.join(qident(c) for c in cols) + ' FROM time_series_feature_state'
           + ('' if not where else ' WHERE ' + ' AND '.join(where)) + ' ORDER BY underlying, state_time')
    con = duckdb.connect(str(DB_PATH), read_only=True)
    try:
        df = con.execute(sql).df()
    finally:
        con.close()
    if df.empty:
        raise RuntimeError('No rows found.')
    df['state_time'] = pd.to_datetime(df['state_time'], utc=True)
    if df.duplicated(['underlying', 'state_time']).any():
        raise RuntimeError('Duplicate asset/timestamp rows.')
    print(f'rows {len(df):,} | assets {sorted(df.underlying.unique())} | {df.state_time.min()} -> {df.state_time.max()}')
    return df.reset_index(drop=True)

def schema_columns():
    con = duckdb.connect(str(DB_PATH), read_only=True)
    try:
        return con.execute('DESCRIBE time_series_feature_state').df()
    finally:
        con.close()

df = load_frame(['futures_index_price'], START, END, ASSETS)
df = df[df.state_time < ow.VRP_CONFIRMATION_START].reset_index(drop=True)
grid = vb.build_grid(df)
jres = jc.run(grid)
excluded = jres['ticks'].loc[jres['ticks'].tick_artifact, 'underlying'].tolist()
panel, S = jres['panel'], jres['S']
used = [a for a in panel.assets if a not in excluded]
I = jc.index_panel(grid, panel)
t0 = time.time()
outcomes = we.origin_outcomes(panel, S, ow.backward_bipower(panel, S), I, used)
print(f'{len(outcomes):,} origins, {outcomes.session.nunique()} sessions, '
      f'{outcomes.groupby(["session","underlying"]).ngroups} session-assets ({time.time()-t0:.0f}s)')
tails = we.tail_table(outcomes)
display(tails.round(4))


rows 42,336 | assets ['AVAX', 'BTC', 'ETH', 'HYPE', 'SOL', 'TRX', 'XRP'] | 2026-09-19 08:00:00+00:00 -> 2026-10-10 07:55:00+00:00
27,986 origins, 18 sessions, 126 session-assets (3s)


,group,n_origins,n_session_assets,u_mean,u_sd,log_k_mean,log_k_sd,P_abs_u_gt_1,normal_1,distinct_session_assets_gt_1,P_abs_u_gt_2,normal_2,distinct_session_assets_gt_2,P_abs_u_gt_3,normal_3,distinct_session_assets_gt_3,P_abs_u_gt_4,normal_4,distinct_session_assets_gt_4,P_abs_u_gt_5,normal_5,distinct_session_assets_gt_5
0,ALL,27986,126,0.0306,1.1757,0.1725,0.7869,0.3019,0.3173,124,0.0878,0.0455,60,0.0285,0.0027,29,0.0063,0.0001,12,0.0029,0.0,5
1,AVAX,3998,18,0.0322,1.0961,0.1337,0.6839,0.2979,0.3173,18,0.0755,0.0455,9,0.0170,0.0027,4,0.0015,0.0001,2,0.0003,0.0,1
2,BTC,3998,18,0.0033,1.2310,0.2539,0.8741,0.3202,0.3173,18,0.1228,0.0455,11,0.0318,0.0027,7,0.0065,0.0001,1,0.0000,0.0,0
3,ETH,3998,18,-0.0023,1.3858,0.2537,0.9350,0.3089,0.3173,17,0.0998,0.0455,7,0.0518,0.0027,5,0.0225,0.0001,4,0.0150,0.0,2
4,HYPE,3998,18,0.2012,1.0561,0.1064,0.6435,0.2904,0.3173,17,0.0640,0.0455,9,0.0210,0.0027,3,0.0008,0.0001,2,0.0000,0.0,0
5,SOL,3998,18,-0.0458,1.0268,0.1528,0.7136,0.2899,0.3173,18,0.0535,0.0455,10,0.0133,0.0027,3,0.0028,0.0001,1,0.0003,0.0,1
6,TRX,3998,18,-0.0060,1.3821,0.2164,0.7510,0.3454,0.3173,18,0.1506,0.0455,8,0.0620,0.0027,4,0.0098,0.0001,1,0.0050,0.0,1
7,XRP,3998,18,0.0320,0.9596,0.0905,0.8452,0.2604,0.3173,18,0.0485,0.0455,6,0.0028,0.0027,3,0.0005,0.0001,1,0.0000,0.0,0
8,"TTE (120.0, 240.0]",3003,126,0.2321,1.1048,0.0251,0.7522,0.3137,0.3173,73,0.0839,0.0455,24,0.0220,0.0027,9,0.0017,0.0001,2,0.0003,0.0,1
9,"TTE (240.0, 480.0]",6048,126,0.0815,1.1033,0.1347,0.7542,0.2970,0.3173,88,0.0544,0.0455,25,0.0152,0.0027,6,0.0104,0.0001,2,0.0063,0.0,2


In [3]:
q = pd.read_csv(IN10)
q = q[q.underlying.isin(used)].reset_index(drop=True)
prim = ow.primary_sample(q, excluded).reset_index(drop=True)
print('Module 10 primary rows:', len(prim), '| sessions:', prim.session.nunique())
t0 = time.time()
res_emp = we.primary(prim, outcomes, kind='emp', n_boot=N_BOOT)
print(f'PRIMARY empirical benchmark ({time.time()-t0:.0f}s):', json.dumps(res_emp, indent=2, default=float))


Module 10 primary rows: 14805 | sessions: 19
PRIMARY empirical benchmark (307s): {
  "n_rows": 14805,
  "n_sessions": 19,
  "M": 1.0499518652785598,
  "ci_low": 0.4762631831000336,
  "ci_high": 2.3416497282362965,
  "verdict": "RICH",
  "bootstrap_verdict": "RICH",
  "share_model_price_zero": 0.0,
  "n_boot": 1000
}


In [ ]:
t0 = time.time()
res_mix = we.primary(prim, outcomes, kind='mix', n_boot=N_BOOT_MIX)
print(f'variance mixture ({time.time()-t0:.0f}s):', json.dumps(res_mix, indent=2, default=float))
prim['lr_emp'] = we.price_loso(prim, outcomes, kind='emp')
prim['lr_emp_tte'] = we.price_loso(prim, outcomes, kind='emp', by_tte=True)
fin = lambda c: prim[np.isfinite(prim[c])]
tte_cond = ow.session_median_stat(fin('lr_emp_tte'), 'lr_emp_tte', n_boot=1000)
tte_cond['share_model_price_zero'] = float(np.isposinf(prim['lr_emp_tte']).mean())
print('TTE-conditioned pool (quote-only bootstrap):', json.dumps(tte_cond, indent=2, default=float))
by_asset = pd.DataFrame([{'underlying': u, **ow.session_median_stat(g[np.isfinite(g.lr_emp)], 'lr_emp', n_boot=1000),
                          'share_model_price_zero': float(np.isposinf(g.lr_emp).mean())}
                         for u, g in prim.groupby('underlying')])
display(by_asset.round(4))


In [ ]:
allq = q[~q.floor_bound].reset_index(drop=True)
allq['log_mid_over_emp'] = we.price_loso(allq, outcomes, kind='emp')
allq['emp_zero'] = np.isposinf(allq['log_mid_over_emp'])
fq = allq[np.isfinite(allq['log_mid_over_emp'])]
buckets = we.bucket_table(fq.assign(floor_bound=False), 'log_mid_over_emp')
zero = allq.groupby(['z_bucket', 'tte_bucket'])['emp_zero'].mean().rename('share_model_price_zero').reset_index()
buckets = buckets.merge(zero, on=['z_bucket', 'tte_bucket'], how='left')
display(buckets.round(4))
display(buckets.pivot(index='z_bucket', columns='tte_bucket', values='M').round(3))
cmp = pd.DataFrame({'model': ['Module 10 jump model', 'empirical (primary)', 'variance mixture', 'empirical TTE-conditioned'],
                    'M': [prim.log_mid_over_jump.median(), res_emp['M'], res_mix['M'], tte_cond['M']]})
display(cmp.round(3))


In [ ]:
outcomes.to_csv(OUT / '10b_origin_outcomes.csv.gz', index=False, compression='gzip')
tails.to_csv(OUT / '10b_tail_frequencies.csv', index=False)
by_asset.to_csv(OUT / '10b_primary_by_asset.csv', index=False)
buckets.to_csv(OUT / '10b_bucket_table.csv', index=False)
decision = {'primary_empirical_z_2_5': res_emp, 'variance_mixture': res_mix, 'tte_conditioned_empirical': tte_cond,
            'module10_jump_model_M_same_rows': float(prim.log_mid_over_jump.median()), 'excluded_assets': excluded}
(OUT / '10b_decision.json').write_text(json.dumps(decision, indent=2, default=float))
meta = {'version': we.VERSION, 'module': '10b_model_free_wing_benchmark', 'consumes': '10/10_priced_quotes.csv.gz (Module 10 primary rows, D)',
        'n_boot': N_BOOT, 'n_boot_mix': N_BOOT_MIX, 'q_nodes': we.Q_NODES, 'tail_raw_share': we.TAIL_RAW_SHARE,
        'n_origins': int(len(outcomes)), 'first_time': str(df.state_time.min()), 'last_time': str(df.state_time.max())}
(OUT / '10b_run_metadata.json').write_text(json.dumps(meta, indent=2, default=float))
print(json.dumps(decision, indent=2, default=float))
